In [0]:
"""
id: downtown_seattle_building_coverage
template: source
name: downtown_seattle_building_coverage
position:
  x: 0
  y: 0
description:
  text: Load data from the specified table downtown_seattle_building_coverage.
  hash: a352b4ca
previewCodeHash: 5bef34821b3c3c26
previewMode: "1000"
config:
  table_source:
    tableName: cmegdemos_catalog.network_analytics_enablement.downtown_seattle_building_coverage
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "cmegdemos_catalog.network_analytics_enablement.downtown_seattle_building_coverage"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["downtown_seattle_building_coverage.data"] = out["data"]

In [0]:
"""
id: count_by_tower_type
template: aggregate
name: count_by_tower_type
position:
  x: 300
  y: 0
description:
  text: Group data by nearest tower type and count the number of entries for each type.
  hash: 6769f786
previewCodeHash: 96a4534815299fa7
previewMode: "1000"
config:
  group_bys:
    - expr: nearest_tower_type
      type: expr
  aggregations:
    - columnExpr:
        expr: nearest_tower_type
        type: expr
      fn: COUNT
      alias: count
input:
  - node: downtown_seattle_building_coverage
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        # Skip pass-through columns that duplicate a group-by column;
        # groupBy() already includes them in the output.
        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            col = agg_fn(raw_expr)
        elif fn == "-" or fn == "_":
            col = F.col(raw_expr)
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "nearest_tower_type",
            "type": "expr"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "nearest_tower_type",
                "type": "expr"
            },
            "fn": "COUNT",
            "alias": "count",
            "withAsKeyword": None
        }
    ]
}
inputs = {
    "data": ctx["downtown_seattle_building_coverage.data"]
}
out = run(config, inputs, spark)
ctx["count_by_tower_type.aggregated_data"] = out["aggregated_data"]

In [0]:
"""
id: cmegdemos_catalog.default.output_2
template: output
name: cmegdemos_catalog.network_analytics_enablement.output_2_demo
position:
  x: 600
  y: 0
description:
  text: Save data to a specific table, replacing any existing data.
  hash: cf93dacf
previewCodeHash: eb927ac953b7da74
previewMode: "1000"
config:
  catalog: cmegdemos_catalog
  schema: network_analytics_enablement
  table_name: output_2_demo
input:
  - node: count_by_tower_type
    input_port: data
    output_port: aggregated_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "cmegdemos_catalog",
    "schema": "network_analytics_enablement",
    "table_name": "output_2_demo"
}
inputs = {
    "data": ctx["count_by_tower_type.aggregated_data"]
}
out = run(config, inputs, spark)